# Pipeline

In [1]:
#from challenge.utils.camera import CameraDisplay
import time
import cv2
now = time.time()

In [3]:
from challenge.tinyyolov2 import TinyYoloV2
from challenge.utils.yolo import nms, filter_boxes
from challenge.utils.viz import display_result
import torch
from typing import List
import numpy as np

## Load TinyYOLO

In [4]:
# make an instance with 20 classes as output
net = TinyYoloV2(num_classes=20)

# load pretrained weights
sd = torch.load("../models/voc_pretrained.pt")
net.load_state_dict(sd)

#put network in evaluation mode
net.eval()

TinyYoloV2(
  (pad): ReflectionPad2d((0, 1, 0, 1))
  (conv1): Conv2d(3, 16, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
  (bn1): BatchNorm2d(16, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
  (conv2): Conv2d(16, 32, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
  (bn2): BatchNorm2d(32, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
  (conv3): Conv2d(32, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
  (bn3): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
  (conv4): Conv2d(64, 128, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
  (bn4): BatchNorm2d(128, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
  (conv5): Conv2d(128, 256, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
  (bn5): BatchNorm2d(256, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
  (conv6): Conv2d(256, 512, kernel_size=(3, 3), stride=

## Boxes + NMS

In [5]:
from challenge.utils.viz import num_to_class


In [6]:

def displayBoxes(frame, output: List[torch.tensor]) -> None:
    pad = 0
    # frame shape is (320,320,3)
    #frame_copy = frame
    frame = np.pad(frame, ((pad, pad), (pad, pad), (0, 0)), mode='constant')
    
    img_shape = 320
        
    if output:
        bboxes = torch.stack(output, dim=0)
        for i in range(bboxes.shape[1]):
            if bboxes[0,i,-1] >= 0:
                # 0 - x center
                # 1 - y center
                # 2 - Width
                # 3 - Height
                # 4 - confidence
                # 5 - class idx
                cx = int(bboxes[0,i,0]*img_shape - bboxes[0,i,2]*img_shape/2) + pad
                cy = int(bboxes[0,i,1]*img_shape - bboxes[0,i,3]*img_shape/2) + pad

                w = int(bboxes[0,i,2]*img_shape)
                h = int(bboxes[0,i,3]*img_shape)
                
                
                start_point = (
                    #int(bboxes[0,i,0]*img_shape- bboxes[0,i,2]*img_shape) + pad,
                    #int(bboxes[0,i,2]*img_shape) + pad
                    int(bboxes[0,i,0]*img_shape - bboxes[0,i,2]*img_shape/2) - pad,
                    int(bboxes[0,i,1]*img_shape - bboxes[0,i,3]*img_shape/2) - pad
                    #int(cx - w/2),
                    #int(cy - h/2)
                )
                end_point = (
                    #int(bboxes[0,i,0]*img_shape - bboxes[0,i,2]*img_shape) + pad,
                    #int(bboxes[0,i,2]*img_shape - bboxes[0,i,3]*img_shape) + pad
                    int(bboxes[0,i,0]*img_shape + bboxes[0,i,2]*img_shape/2) + pad,
                    int(bboxes[0,i,1]*img_shape + bboxes[0,i,3]*img_shape/2) + pad
                    #int(cx + w/2),
                    #int(cy + h/2)
                )
                
                
                color = (255, 0, 0) 
                print(num_to_class(int(bboxes[0,i,5])))
                #print(bboxes*img_shape)
                #print(bboxes[0,i,0]*img_shape)
                #print(bboxes[0,i,1]*img_shape)
                #print(bboxes[0,i,2]*img_shape)
                #print(bboxes[0,i,3]*img_shape)
                print((start_point, end_point))
                
                cv2.rectangle(frame, start_point, end_point, color, 2) 

    return frame

In [7]:
def callback(frame):
    global now

    fps = f"{int(1/(time.time() - now))}"
    now = time.time()
        
    # run model
    input = torch.tensor(frame, dtype=torch.float32).permute(2,0,1).view(1,3,320,320)
    output = net(input)
    print(output.shape)
    #filter boxes based on confidence score (class_score*confidence)
    output = filter_boxes(output, 0.1)
    #print(conf)
    
    #filter boxes based on overlap
    output = nms(output, 0.25)
    
    # add boxes
    frame = displayBoxes(frame, output)
    
    
    #cv2.line(frame, (125, 34), (130, 221), (255,0,0), 10) 
    cv2.putText(frame, "fps="+fps, (2, 25), cv2.FONT_HERSHEY_SIMPLEX, 1,
                (100, 255, 0), 2, cv2.LINE_AA)
    
    return frame

## Camera loop

In [8]:
cam = cv2.VideoCapture(0, cv2.CAP_DSHOW)

if(cam.isOpened() == False):
    print("Error opening camera bruv")
else:
    fps = cam.get(5)
    print('Frames per second : ', fps,'FPS')

# Get frame count
# You can replace 7 with CAP_PROP_FRAME_COUNT as well, they are enumerations
frame_count = cam.get(7)
print('Frame count : ', frame_count)
 
image = cv2.imread('./me2.png')
 
while(cam.isOpened()):
    # first element is a bool, the second is frame
    ret, frame = cam.read()
    if ret == True:
        # resize image from webcam res (480,640) to a lower one
        #resized = cv2.resize(frame[:,80:-80], (320,320))#, interpolation = cv2.INTER_AREA)
        resized = cv2.resize(image[:320,:320], (320,320))
        
        resized = callback(resized)
        
        '''        
        global now

        fps = f"{int(1/(time.time() - now))}"
        now = time.time()
        
        input = torch.tensor(resized, dtype=torch.float32).permute(2,0,1).view(1,3,320,320)
        output = net(input)
        output = filter_boxes(output, 0.1)
        
        output = nms(output, 0.25)
        
        resized = displayBoxes(resized, output)
        
        #cv2.line(resized, (125, 34), (130, 221), (255,0,0), 10) 

        
        cv2.putText(resized, "fps="+fps, (2, 25), cv2.FONT_HERSHEY_SIMPLEX, 1,
                    (100, 255, 0), 2, cv2.LINE_AA)
        '''
        
        cv2.imshow('Frame',resized)
        key = cv2.waitKey(20)

        
        if key == ord('q'):
            break
    else:
        break
 
# Release the video capture object
cam.release()
cv2.destroyAllWindows()

Error opening camera bruv
Frame count :  0.0
